In [ ]:
%pip install -U transformers accelerate pillow

## Local Inference on GPU 
Model page: https://huggingface.co/openbmb/MiniCPM-o-4_5

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/openbmb/MiniCPM-o-4_5)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Load model directly
import torch
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_utils import PreTrainedModel

if not hasattr(PreTrainedModel, "all_tied_weights_keys"):
    PreTrainedModel.all_tied_weights_keys = {}

model_path = "openbmb/MiniCPM-o-4_5"
model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    device_map=None,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    init_vision=False,
    init_audio=False,
    init_tts=False,
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
prompt = "Answer briefly: what is 2 plus 2?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.llm(**inputs)
next_token_id = outputs.logits[0, -1].argmax().item()
print("next token =", tokenizer.decode([next_token_id]))
model_parameter_devices = sorted({str(p.device) for p in model.parameters()})
print("model parameter devices =", model_parameter_devices)
